In [1]:
import pandas as pd
import os

In [2]:

# 📁 Ruta donde están tus archivos
ruta = r'C:\Users\loed2001\OneDrive - Nielsen IQ\Desktop\Curso PY\Practicas.py\tipos\Mercado_libre\Datos climatologicos\Datos climatologicos 2'

# 2. Detectar automáticamente todos los CSV
# =========================
archivos = [f for f in os.listdir(ruta) if f.endswith(".csv")]

print(f"Archivos detectados: {len(archivos)}")

dfs = []

for archivo in archivos:
    path = os.path.join(ruta, archivo)
    
    # Leer CSV (forzamos string para evitar errores)
    df = pd.read_csv(path, dtype=str, encoding='latin1')
    
    # 🧠 Extraer municipio desde nombre archivo
    municipio = archivo.replace('.csv', '').split('_')[-1]
    
    # Agregar columna
    df['Municipio'] = municipio
    
    dfs.append(df)

# 🔗 Concatenar todo
df_final = pd.concat(dfs, ignore_index=True)

# 💾 Guardar archivo final
output_path = os.path.join(ruta, 'dataset_concatenado_municipios.csv')
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print("Archivo generado en:", output_path)

Archivos detectados: 18
Archivo generado en: C:\Users\loed2001\OneDrive - Nielsen IQ\Desktop\Curso PY\Practicas.py\tipos\Mercado_libre\Datos climatologicos\Datos climatologicos 2\dataset_concatenado_municipios.csv


In [3]:
df_final


,YEAR,DOY,ALLSKY_SFC_SW_DWN,T2M,QV2M,RH2M,PRECTOTCORR,WS2M,GWETTOP,GWETROOT,GWETPROF,Municipio
0,2023,2,11.71,12.68,5.04,59.31,0.09,1.56,0.43,0.47,0.47,nuevas
1,2023,3,10.99,14.54,6.52,65.5,0.39,1.4,0.41,0.47,0.47,nuevas
2,2023,4,11.8,14.17,7.1,74.74,0.09,2.23,0.41,0.47,0.47,nuevas
3,2023,5,9.47,15.24,7.31,72.33,0.73,1.7,0.4,0.47,0.47,nuevas
4,2023,6,11.38,13.15,5.1,58.38,0.01,2.33,0.4,0.47,0.47,nuevas
...,...,...,...,...,...,...,...,...,...,...,...,...
19723,2025,362,8.84,19.76,5.77,42.61,0,1.64,0.27,0.46,0.46,Ures
19724,2025,363,13.11,21.86,6.76,39.92,0,2.06,0.26,0.46,0.46,Ures
19725,2025,364,9.25,22.25,5.44,31.84,0,3.22,0.25,0.46,0.46,Ures
19726,2025,365,4.2,21.06,6.94,43.77,1.47,2.22,0.25,0.46,0.46,Ures


In [12]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19728 entries, 0 to 19727
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   YEAR               19728 non-null  object
 1   DOY                19728 non-null  object
 2   ALLSKY_SFC_SW_DWN  19728 non-null  object
 3   T2M                19728 non-null  object
 4   QV2M               19728 non-null  object
 5   RH2M               19728 non-null  object
 6   PRECTOTCORR        19728 non-null  object
 7   WS2M               19728 non-null  object
 8   GWETTOP            19728 non-null  object
 9   GWETROOT           19728 non-null  object
 10  GWETPROF           19728 non-null  object
 11  Municipio          19728 non-null  object
dtypes: object(12)
memory usage: 1.8+ MB


In [14]:
# Limpiar espacios por si vienen raros
df_final["YEAR"] = df_final["YEAR"].astype(str).str.strip()
df_final["DOY"] = df_final["DOY"].astype(str).str.strip()

# Convertir a numérico
df_final["YEAR"] = pd.to_numeric(df_final["YEAR"], errors="coerce")
df_final["DOY"] = pd.to_numeric(df_final["DOY"], errors="coerce")

# Revisar si hubo valores inválidos
print(df_final[["YEAR", "DOY"]].isna().sum())

# Crear fecha
df_final["fecha"] = (
    pd.to_datetime(df_final["YEAR"].astype("Int64").astype(str), format="%Y")
    + pd.to_timedelta(df_final["DOY"] - 1, unit="D")
)

# Opcional: sacar mes y día
df_final["MO"] = df_final["fecha"].dt.month
df_final["DY"] = df_final["fecha"].dt.day

# Validar
print(df_final[["YEAR", "DOY", "fecha", "MO", "DY"]].head())


YEAR    0
DOY     0
dtype: int64
   YEAR  DOY      fecha  MO  DY
0  2023    2 2023-01-02   1   2
1  2023    3 2023-01-03   1   3
2  2023    4 2023-01-04   1   4
3  2023    5 2023-01-05   1   5
4  2023    6 2023-01-06   1   6


In [15]:

# 💾 Guardar archivo final
output_path = os.path.join(ruta, 'dataset_concatenado_municipios_final2.csv')
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')


In [7]:
#2. Crear fecha y fecha-hora
# =========================
df_final["fecha"] = pd.to_datetime({
    "year": df_final["YEAR"],
    "day": df_final["DOY"]
})

df_final["año_dia"] = pd.to_datetime(
    df_final["YEAR"].astype(str) + "-" +
    df_final["DOY"].astype(str).str.zfill(2) + " " +
)

SyntaxError: invalid syntax (3679757868.py, line 11)

In [11]:
# =========================
df = df.sort_values(["Municipio", "fecha", "HR"]).reset_index(drop=True)

# =========================
# 4. Crear tabla única de días por municipio
#    (una fila por municipio por día)
# =========================
dias_unicos = (
    df[["Municipio", "fecha"]]
    .drop_duplicates()
    .sort_values(["Municipio", "fecha"])
    .reset_index(drop=True)
)

In [12]:
dias_unicos

,Municipio,fecha
0,Altar,2023-01-02
1,Altar,2023-01-03
2,Altar,2023-01-04
3,Altar,2023-01-05
4,Altar,2023-01-06
...,...,...
19726,victoria,2025-12-28
19727,victoria,2025-12-29
19728,victoria,2025-12-30
19729,victoria,2025-12-31


In [13]:
# 5. Asignar conteo secuencial de días por municipio
# =========================
dias_unicos["dia_global"] = dias_unicos.groupby("Municipio").cumcount() + 1

In [14]:
# 6. Convertir a ciclo de 1 a 150
#    Ejemplo:
#    1->1, 150->150, 151->1, 152->2 ...
# =========================
dias_unicos["dia_ciclo"] = ((dias_unicos["dia_global"] - 1) % 150) + 1

In [15]:
# 7. Función para etapa fenológica + kc del trigo
# =========================
def asignar_etapa_kc_trigo(dia):
    dia = int(dia)

    if 1 <= dia <= 15:
        return pd.Series(["Germinación / Emergencia", 0.30])
    elif 16 <= dia <= 55:
        return pd.Series(["Amacollamiento", 0.55])
    elif 56 <= dia <= 75:
        return pd.Series(["Encañe", 0.85])
    elif 76 <= dia <= 80:
        return pd.Series(["Embuche / Espigamiento", 1.10])
    elif 81 <= dia <= 85:
        return pd.Series(["Floración", 1.15])
    elif 86 <= dia <= 110:
        return pd.Series(["Grano acuoso-lechoso", 1.10])
    elif 111 <= dia <= 130:
        return pd.Series(["Grano masoso", 0.85])
    elif 131 <= dia <= 150:
        return pd.Series(["Madurez fisiológica / Cosecha", 0.35])
    else:
        return pd.Series(["Fuera de ciclo", np.nan])

dias_unicos[["etapa_fenologica", "kc"]] = dias_unicos["dia_ciclo"].apply(asignar_etapa_kc_trigo)

In [ ]:
# 8. Unir la información de regreso al dataset horario
# =========================
df = df.merge(
    dias_unicos[["Municipio", "fecha", "dia_global", "dia_ciclo", "etapa_fenologica", "kc"]],
    on=["Municipio", "fecha"],
    how="left"
)

In [17]:
# 9. Validación rápida
# =========================
print(df[[
    "Municipio", "fecha", "HR", "dia_global",
    "dia_ciclo", "etapa_fenologica", "kc"
]].head(30))

   Municipio      fecha  HR  dia_global  dia_ciclo          etapa_fenologica  \
0      Altar 2023-01-02   0           1          1  Germinación / Emergencia   
1      Altar 2023-01-02   1           1          1  Germinación / Emergencia   
2      Altar 2023-01-02   2           1          1  Germinación / Emergencia   
3      Altar 2023-01-02   3           1          1  Germinación / Emergencia   
4      Altar 2023-01-02   4           1          1  Germinación / Emergencia   
5      Altar 2023-01-02   5           1          1  Germinación / Emergencia   
6      Altar 2023-01-02   6           1          1  Germinación / Emergencia   
7      Altar 2023-01-02   7           1          1  Germinación / Emergencia   
8      Altar 2023-01-02   8           1          1  Germinación / Emergencia   
9      Altar 2023-01-02   9           1          1  Germinación / Emergencia   
10     Altar 2023-01-02  10           1          1  Germinación / Emergencia   
11     Altar 2023-01-02  11           1 

In [18]:
# 10. Guardar resultado
# =========================
df.to_csv("dataset_trigo_municipios_con_ciclos.csv", index=False)

print("Archivo guardado: dataset_trigo_municipios_con_ciclos.csv")

Archivo guardado: dataset_trigo_municipios_con_ciclos.csv


In [20]:
df.head()

,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,T2M,RH2M,QV2M,PRECTOTCORR,PS,WS2M,Municipio,fecha,fecha_hora,dia_global,dia_ciclo,etapa_fenologica,kc
0,2023,1,2,0,0.0,7.12,94.60,6.36,1.81,93.84,3.40,Altar,2023-01-02,2023-01-02 00:00:00,1,1,Germinación / Emergencia,0.3
1,2023,1,2,1,0.0,6.49,96.79,6.23,1.18,93.86,2.75,Altar,2023-01-02,2023-01-02 01:00:00,1,1,Germinación / Emergencia,0.3
2,2023,1,2,2,0.0,6.34,96.70,6.16,1.05,93.85,2.38,Altar,2023-01-02,2023-01-02 02:00:00,1,1,Germinación / Emergencia,0.3
3,2023,1,2,3,0.0,6.35,95.99,6.12,0.77,93.84,1.90,Altar,2023-01-02,2023-01-02 03:00:00,1,1,Germinación / Emergencia,0.3
4,2023,1,2,4,0.0,6.41,95.13,6.09,0.60,93.84,1.65,Altar,2023-01-02,2023-01-02 04:00:00,1,1,Germinación / Emergencia,0.3


In [21]:
df = df.replace(-999, np.nan)

In [22]:
df_daily = (
    df.groupby(["Municipio", "fecha"], as_index=False)
      .agg(
          t_mean=("T2M", "mean"),
          t_max=("T2M", "max"),
          t_min=("T2M", "min"),
          rh_mean=("RH2M", "mean"),
          ws2m_mean=("WS2M", "mean"),
          ps_mean=("PS", "mean"),
          rs_sum=("ALLSKY_SFC_SW_DWN", "sum"),
          lluvia_sum=("PRECTOTCORR", "sum"),
          dia_ciclo=("dia_ciclo", "first"),
          etapa_fenologica=("etapa_fenologica", "first"),
          kc=("kc", "first")
      )
)

In [23]:


def saturation_vapor_pressure(T):
    return 0.6108 * np.exp((17.27 * T) / (T + 237.3))

def slope_vapor_pressure_curve(T):
    return (4098 * saturation_vapor_pressure(T)) / ((T + 237.3) ** 2)

df_daily["es_tmax"] = saturation_vapor_pressure(df_daily["t_max"])
df_daily["es_tmin"] = saturation_vapor_pressure(df_daily["t_min"])
df_daily["es"] = (df_daily["es_tmax"] + df_daily["es_tmin"]) / 2

df_daily["ea"] = (df_daily["rh_mean"] / 100) * df_daily["es"]
df_daily["delta"] = slope_vapor_pressure_curve(df_daily["t_mean"])

# Si PS está en kPa, como parece por tus valores ~94
df_daily["gamma"] = 0.000665 * df_daily["ps_mean"]

# Radiación neta simplificada
df_daily["rn"] = 0.77 * df_daily["rs_sum"]

df_daily["et0"] = (
    (
        0.408 * df_daily["delta"] * df_daily["rn"] +
        df_daily["gamma"] * (900 / (df_daily["t_mean"] + 273)) *
        df_daily["ws2m_mean"] * (df_daily["es"] - df_daily["ea"])
    )
    /
    (
        df_daily["delta"] + df_daily["gamma"] * (1 + 0.34 * df_daily["ws2m_mean"])
    )
).clip(lower=0)

df_daily["etc"] = df_daily["et0"] * df_daily["kc"]

In [24]:
df = df.merge(
    df_daily[[
        "Municipio", "fecha", "et0", "etc",
        "dia_ciclo", "etapa_fenologica", "kc"
    ]],
    on=["Municipio", "fecha"],
    how="left",
    suffixes=("", "_daily")
)

In [ ]:
# Ordenar
df_daily = df.sort_values(["Municipio", "fecha"]).copy()

In [26]:
# Parámetros globales del prototipo
HUMEDAD_INICIAL = 60.0
HUMEDAD_MIN = 5.0
HUMEDAD_MAX = 100.0

ALPHA_LLUVIA = 0.8   # proporción que infiltra
BETA_ETC = 1.0       # pérdida por demanda del cultivo
GAMMA_DRENAJE = 0.4  # drenaje si supera capacidad alta
UMBRAL_DRENAJE = 85.0

humedad_estim = []

for municipio, sub in df_daily.groupby("Municipio", sort=False):
    h = HUMEDAD_INICIAL
    
    for _, row in sub.iterrows():
        lluvia = 0 if pd.isna(row["lluvia_sum"]) else row["lluvia_sum"]
        etc = 0 if pd.isna(row["etc"]) else row["etc"]
        
        # Aporte y consumo
        h = h + ALPHA_LLUVIA * lluvia - BETA_ETC * etc
        
        # Drenaje si se "satura"
        if h > UMBRAL_DRENAJE:
            exceso = h - UMBRAL_DRENAJE
            h = h - GAMMA_DRENAJE * exceso
        
        # Limitar rango
        h = max(HUMEDAD_MIN, min(HUMEDAD_MAX, h))
        
        humedad_estim.append(round(h, 2))

df_daily["humedad_suelo_estimada"] = humedad_estim

In [27]:
def clasificar_humedad(h):
    if h < 30:
        return "Baja"
    elif h < 60:
        return "Media"
    else:
        return "Alta"

def recomendar_riego(h, etc):
    if h < 30:
        return round((35 - h) * 0.6 + etc, 2)
    elif h < 45:
        return round((45 - h) * 0.3 + etc * 0.5, 2)
    return 0.0

df_daily["nivel_humedad"] = df_daily["humedad_suelo_estimada"].apply(clasificar_humedad)
df_daily["agua_recomendada_mm"] = df_daily.apply(
    lambda x: recomendar_riego(x["humedad_suelo_estimada"], x["etc"]),
    axis=1
)
df_daily["regar_hoy"] = (df_daily["agua_recomendada_mm"] > 0).astype(int)

In [28]:
df = df.merge(
    df_daily[[
        "Municipio", "fecha",
        "humedad_suelo_estimada",
        "nivel_humedad",
        "agua_recomendada_mm",
        "regar_hoy"
    ]],
    on=["Municipio", "fecha"],
    how="left"
)

In [30]:
df.columns

Index(['YEAR', 'MO', 'DY', 'HR', 'ALLSKY_SFC_SW_DWN', 'T2M', 'RH2M', 'QV2M',
       'PRECTOTCORR', 'PS', 'WS2M', 'Municipio', 'fecha', 'fecha_hora',
       'dia_global', 'dia_ciclo', 'etapa_fenologica', 'kc', 'et0', 'etc',
       'dia_ciclo_daily', 'etapa_fenologica_daily', 'kc_daily',
       'humedad_suelo_estimada', 'nivel_humedad', 'agua_recomendada_mm',
       'regar_hoy'],
      dtype='object')

In [31]:
df.to_csv("dataset_trigo_municipios_con_ciclos_final.csv", index=False)

print("Archivo guardado: dataset_trigo_municipios_con_ciclos.csv")

Archivo guardado: dataset_trigo_municipios_con_ciclos.csv
